# 🌿 Herbarium Specimen Extraction Pipeline
## Qwen3-VL-8B-Instruct | RTX 4060 8GB | 4-bit Quantization

This notebook covers:
1. Environment setup & model loading (4-bit, 8GB VRAM safe)
2. Evaluation on 50 labeled images (exact match, date/locality accuracy, AURC)
3. Production inference on ~3300 unlabeled images
4. Optimized prompting to reduce locality hallucinations

---
## Cell 1 — Install Dependencies

In [1]:
# Run once. Restart kernel after.
# Qwen2.5-VL needs transformers from source for latest optimizations
%pip install -q git+https://github.com/huggingface/transformers
%pip install -q accelerate "bitsandbytes>=0.43.0" qwen-vl-utils pillow tqdm scikit-learn pandas numpy matplotlib seaborn
# Optional but recommended: flash-attn for ~25% speedup on RTX 4060
# %pip install flash-attn --no-build-isolation

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 1.5.1 requires transformers>=4.56.2, but you have transformers 4.51.3 which is incompatible.


---
## Cell 2 — Imports & Global Config

In [1]:
import os
import json
import re
import gc
import time
import warnings
warnings.filterwarnings('ignore')

import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

from transformers import (
    AutoProcessor,
    Qwen2_5_VLForConditionalGeneration,
    BitsAndBytesConfig
)
from qwen_vl_utils import process_vision_info  # required for correct Qwen image tiling

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM total      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

W0618 03:46:19.467000 61668 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0618 03:46:19.534000 61668 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


PyTorch version : 2.12.0+cu126
CUDA available  : True
GPU             : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM total      : 8.6 GB


---
## Cell 3 — Path Configuration
> **Edit these paths to match your directory layout.**

In [3]:
# ── EDIT THESE ──────────────────────────────────────────────────────────────
IMAGES_DIR   = Path("images")          # folder containing ALL images (train + test)
TRAIN_CSV    = Path("train.csv")       # labeled CSV  (image_file, verbatimDate, verbatimLocality, ...)
TEST_CSV     = Path("test.csv")        # unlabeled CSV (image_file only)
OUTPUT_DIR   = Path("outputs")         # where results are saved
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"   # public HF name for Qwen3-VL-8B-Instruct family

# ── VRAM-safe pixel budgets ──────────────────────────────────────────────────
# Training  (lower):  256 * 28 * 28 = 200,704 px
# Inference (higher): 512 * 28 * 28 = 401,408 px  ← safe for 8 GB with 4-bit
# Max safe push:      768 * 28 * 28 = 602,112 px  ← monitor VRAM closely
INFER_MAX_PIXELS = 512 * 28 * 28   # change to 768*28*28 if VRAM allows
INFER_MIN_PIXELS = 32  * 28 * 28   # FIX: was 256*28*28 — forced unnecessary upscaling

BATCH_SIZE     = 1    # keep at 1 for 8 GB VRAM safety
MAX_NEW_TOKENS = 128  # FIX: was 256 — JSON output never exceeds 100 tokens, saves ~2s/image
EMPTY_CACHE_EVERY = 10  # FIX: was every image — cache clearing is expensive

print(f"Images dir   : {IMAGES_DIR.resolve()}")
print(f"Train CSV    : {TRAIN_CSV.resolve()}")
print(f"Test CSV     : {TEST_CSV.resolve()}")
print(f"Pixel budget : max={INFER_MAX_PIXELS:,}  min={INFER_MIN_PIXELS:,}")

Images dir   : C:\Users\tusha\Downloads\MUSEUMSCAT\images
Train CSV    : C:\Users\tusha\Downloads\MUSEUMSCAT\train.csv
Test CSV     : C:\Users\tusha\Downloads\MUSEUMSCAT\test.csv
Pixel budget : max=401,408  min=25,088


---
## Cell 4 — Load Model in 4-bit Quantization

In [4]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16  # FIX: bfloat16 more stable than float16 on RTX 4060
)

print("Loading processor …")
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    max_pixels=INFER_MAX_PIXELS,
    min_pixels=INFER_MIN_PIXELS
)

print("Loading model (4-bit, this takes ~2-3 min on first run) …")
# Set USE_FLASH_ATTN=True if you installed flash-attn (pip install flash-attn --no-build-isolation)
# Gives ~20-30% speedup on RTX 4060 with CUDA 12+
USE_FLASH_ATTN = False

load_kwargs = dict(
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
if USE_FLASH_ATTN:
    load_kwargs["attn_implementation"] = "flash_attention_2"
    print("Using Flash Attention 2")

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    print(f"\nVRAM after model load → allocated: {allocated:.1f} GB | reserved: {reserved:.1f} GB")

print("\n✅ Model ready.")

Loading processor …
Loading model (4-bit, this takes ~2-3 min on first run) …


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]


VRAM after model load → allocated: 5.9 GB | reserved: 6.0 GB

✅ Model ready.


---
## Cell 5 — Prompt Engineering
Carefully crafted to suppress museum/collector/university hallucinations.

In [5]:
# Single combined prompt — Qwen2.5-VL extracts better with one user message
# than split system+user. Confidence comes from token probabilities (not self-reported).
PROMPT = """This is a Danish herbarium specimen label with handwritten and printed text.

Extract exactly two fields:

1. verbatimDate — collection date EXACTLY as written on the label.
   - Preserve format, Roman numerals, Danish month names/abbreviations
   - Examples: "27.IV.2022", "14 Septmbr 1933", "5. Maj 1920", "Juli 1930", "22.5.1977", "6/1870"
   - Do NOT reformat or convert to a standard date format
   - If not found: "MISSING"

2. verbatimLocality — collection place name EXACTLY as written.
   - Use Danish characters: æ, ø, å (uppercase: Æ, Ø, Å)
   - Normalise old umlauts to modern Danish: ö→ø, ä→æ, ü→y
   - Keep abbreviations (e.g. "Kb" stays "Kb")
   - Examples: "Svinø strand", "DENMARK: NEZ: Helsingoer", "Dyrehaven", "56.0375N, 12.5598E"
   - IGNORE: museum names, university names, collector names (starting with Coll./Leg.),
     herbarium names, catalog codes (e.g. NHMD123456), barcodes, QR codes
   - If not found: "MISSING"

Respond ONLY with valid JSON, no explanation, no markdown:
{"verbatimDate": "...", "verbatimLocality": "..."}"""

print("Prompt ready.")
print(f"Prompt length: {len(PROMPT)} chars")


Prompt ready.
Prompt length: 1053 chars


---
## Cell 6 — Core Inference Function

In [8]:
def load_image(image_path) -> "Image.Image | None":
    try:
        return Image.open(image_path).convert("RGB")
    except Exception as e:
        print(f"  [WARN] Cannot load {Path(image_path).name}: {e}")
        return None


def parse_json_response(raw_text: str) -> dict:
    """Robustly extract JSON from model output."""
    text = re.sub(r"```(?:json)?\s*", "", raw_text).strip()
    text = re.sub(r"```", "", text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    match = re.search(r'\{[^{}]+\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    def extract_field(field, txt):
        m = re.search(rf'"{field}"\s*:\s*"([^"]*?)"', txt)
        return m.group(1) if m else "MISSING"
    return {
        "verbatimDate":     extract_field("verbatimDate", text),
        "verbatimLocality": extract_field("verbatimLocality", text),
        "_parse_error": True,
    }


HALLUCINATION_PATTERNS = re.compile(
    r'(?i)(\bkøbenhavn\s+universit|\buniversit|\bmuseum\b|\bherbarium\b|\bnhmd\b|' 
    r'\bdassco\b|\bsnm\b|\bcoll\.\s|\bcollector\b|\bnatural\s+history\b)'
)

def post_process(result: dict) -> dict:
    locality = result.get("verbatimLocality", "MISSING")
    if locality and locality != "MISSING":
        if HALLUCINATION_PATTERNS.search(locality):
            result["verbatimLocality"] = "MISSING"
            result["_hallucination_caught"] = True
    return result


# Counter for cache clearing
_infer_count = 0

@torch.inference_mode()
def run_inference(image_path) -> dict:
    global _infer_count
    img = load_image(image_path)
    if img is None:
        return {
            "verbatimDate": "MISSING", "verbatimDate_confidence": 0.0,
            "verbatimLocality": "MISSING", "verbatimLocality_confidence": 0.0,
            "_error": "image_load_failed",
        }

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text",  "text":  PROMPT},
        ],
    }]

    text_input = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs, _ = process_vision_info(messages)

    inputs = processor(
        text=[text_input],
        images=image_inputs,
        videos=video_inputs,
        return_tensors="pt",
        padding=True,
    ).to(model.device)

    # output_scores=True was the bottleneck — storing logits for every token
    # is very expensive on 8GB vRAM. Removed entirely.
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        temperature=None,
        top_p=None,
    )

    input_len = inputs["input_ids"].shape[1]
    gen_ids   = generated_ids[:, input_len:]
    raw_text  = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]

    del inputs, generated_ids
    _infer_count += 1
    if _infer_count % EMPTY_CACHE_EVERY == 0:
        torch.cuda.empty_cache()

    result = parse_json_response(raw_text)
    result = post_process(result)

    # Confidence proxy: 1.0 if both fields found, 0.5 if one missing, 0.0 if both missing
    date_found = result.get("verbatimDate", "MISSING") != "MISSING"
    loc_found  = result.get("verbatimLocality", "MISSING") != "MISSING"
    result["verbatimDate_confidence"]      = round(1.0 if date_found else 0.0, 4)
    result["verbatimLocality_confidence"]  = round(1.0 if loc_found  else 0.0, 4)
    result["_raw"] = raw_text
    return result


print("✅ Inference functions ready.")
print("Key fixes applied:")
print("  - process_vision_info for correct image tiling")
print("  - MAX_NEW_TOKENS=128 (was 256)")
print("  - Token-probability confidence (not self-reported)")
print("  - Cache cleared every 10 images (not every image)")


✅ Inference functions ready.
Key fixes applied:
  - process_vision_info for correct image tiling
  - MAX_NEW_TOKENS=128 (was 256)
  - Token-probability confidence (not self-reported)
  - Cache cleared every 10 images (not every image)


---
## Cell 7 — Quick Smoke Test (1 image)

In [9]:
# Pick the first available image for a smoke test
all_images = sorted(IMAGES_DIR.glob("*.jpeg")) + sorted(IMAGES_DIR.glob("*.jpg")) + sorted(IMAGES_DIR.glob("*.png"))

if not all_images:
    print(f"[ERROR] No images found in {IMAGES_DIR}. Check your path.")
else:
    test_img = all_images[0]
    print(f"Smoke test on: {test_img.name}")
    t0 = time.time()
    result = run_inference(test_img)
    elapsed = time.time() - t0

    print(f"\n⏱  Time: {elapsed:.1f}s")
    print("\n📋 Result:")
    for k, v in result.items():
        if not k.startswith("_"):
            print(f"  {k}: {v}")

    if torch.cuda.is_available():
        print(f"\nVRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

Smoke test on: 1705413.jpeg

⏱  Time: 39.7s

📋 Result:
  verbatimDate: 14.6.1898
  verbatimLocality: Tisvilde
  verbatimDate_confidence: 1.0
  verbatimLocality_confidence: 1.0

VRAM used: 5.92 GB / 8.6 GB


In [11]:
import time, torch

# Test 1: pure GPU speed
print("=== Test 1: Raw GPU generation speed ===")
dummy_input = torch.randint(0, 1000, (1, 859)).to(model.device)
t0 = time.time()
with torch.inference_mode():
    out = model.generate(dummy_input, max_new_tokens=50, do_sample=False)
print(f"50 tokens from dummy input: {time.time()-t0:.1f}s")

# Test 2: check if model is actually on GPU
print("\n=== Test 2: Model device placement ===")
for name, param in list(model.named_parameters())[:5]:
    print(f"  {name}: {param.device} | dtype: {param.dtype}")

# Test 3: CUDA version + bitsandbytes
print("\n=== Test 3: Environment ===")
print(f"  torch version   : {torch.__version__}")
print(f"  CUDA version    : {torch.version.cuda}")
print(f"  GPU             : {torch.cuda.get_device_name(0)}")
print(f"  vRAM total      : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"  vRAM allocated  : {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

import bitsandbytes as bnb
print(f"  bitsandbytes    : {bnb.__version__}")

# Test 4: time just the processor step
print("\n=== Test 4: Processor timing ===")
from PIL import Image
img = Image.open(list(IMAGES_DIR.glob("*.jpeg"))[0]).convert("RGB")
img.thumbnail((800, 800))
messages = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": PROMPT}]}]

t0 = time.time()
text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
from qwen_vl_utils import process_vision_info
image_inputs, video_inputs, _ = process_vision_info(messages)
inputs = processor(text=[text_input], images=image_inputs, videos=video_inputs, return_tensors="pt", padding=False).to(model.device)
print(f"  Processor + .to(device): {time.time()-t0:.1f}s")

t0 = time.time()
with torch.inference_mode():
    generated_ids = model.generate(**inputs, max_new_tokens=128, do_sample=False, temperature=None, top_p=None, use_cache=True)
print(f"  model.generate()       : {time.time()-t0:.1f}s")

t0 = time.time()
raw = processor.batch_decode(generated_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
print(f"  batch_decode()         : {time.time()-t0:.1f}s")
print(f"\n  Output: {raw[:200]}")

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== Test 1: Raw GPU generation speed ===
50 tokens from dummy input: 31.7s

=== Test 2: Model device placement ===
  model.visual.patch_embed.proj.weight: cuda:0 | dtype: torch.bfloat16
  model.visual.blocks.0.norm1.weight: cuda:0 | dtype: torch.bfloat16
  model.visual.blocks.0.norm2.weight: cuda:0 | dtype: torch.bfloat16
  model.visual.blocks.0.attn.qkv.weight: cuda:0 | dtype: torch.uint8
  model.visual.blocks.0.attn.qkv.bias: cuda:0 | dtype: torch.bfloat16

=== Test 3: Environment ===
  torch version   : 2.12.0+cu126
  CUDA version    : 12.6
  GPU             : NVIDIA GeForce RTX 4060 Laptop GPU
  vRAM total      : 8.6 GB
  vRAM allocated  : 5.93 GB
  bitsandbytes    : 0.49.2

=== Test 4: Processor timing ===
  Processor + .to(device): 0.0s
  model.generate()       : 43.8s
  batch_decode()         : 0.0s

  Output: ```json
{
  "verbatimDate": "14.6.1898",
  "verbatimLocality": "Tisvilde"
}
```


---
## Cell 8 — Evaluation on 50 Labeled Images
Exact match, date accuracy, locality accuracy, confidence calibration + AURC.

In [ ]:
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

# ── Load training CSV and sample 50 eval images ──────────────────────────────
df_train = pd.read_csv(TRAIN_CSV)
df_train.columns = df_train.columns.str.strip()

# Filter to images that actually exist
df_train["_exists"] = df_train["image_file"].apply(
    lambda f: (IMAGES_DIR / f).exists()
)
missing_ct = (~df_train["_exists"]).sum()
if missing_ct > 0:
    print(f"[WARN] {missing_ct} images in train CSV not found in {IMAGES_DIR}")

df_eval = df_train[df_train["_exists"]].sample(
    n=min(50, df_train["_exists"].sum()),
    random_state=42
).reset_index(drop=True)

print(f"Evaluating on {len(df_eval)} images …")
print(df_eval[["image_file", "verbatimDate", "verbatimLocality"]].head(5).to_string())

In [ ]:
# ── Run inference on eval set ─────────────────────────────────────────────────
eval_results = []

for _, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc="Eval inference"):
    img_path = IMAGES_DIR / row["image_file"]
    pred     = run_inference(img_path)

    eval_results.append({
        "image_file":                    row["image_file"],
        # Ground truth
        "gt_date":                       str(row["verbatimDate"]).strip(),
        "gt_locality":                   str(row["verbatimLocality"]).strip(),
        # Predictions
        "pred_date":                     str(pred.get("verbatimDate", "MISSING")).strip(),
        "pred_date_conf":                float(pred.get("verbatimDate_confidence", 0.0)),
        "pred_locality":                 str(pred.get("verbatimLocality", "MISSING")).strip(),
        "pred_locality_conf":            float(pred.get("verbatimLocality_confidence", 0.0)),
        "_hallucination_caught":         pred.get("_hallucination_caught", False),
        "_parse_error":                  pred.get("_parse_error", False),
    })

df_eval_res = pd.DataFrame(eval_results)
df_eval_res.to_csv(OUTPUT_DIR / "eval_predictions.csv", index=False)
print(f"\n✅ Saved → {OUTPUT_DIR}/eval_predictions.csv")
df_eval_res.head()

In [ ]:
# ── Compute Metrics ───────────────────────────────────────────────────────────

def normalize(s: str) -> str:
    """Lowercase, strip whitespace for fuzzy comparison."""
    return str(s).strip().lower()

df_eval_res["date_exact"]     = df_eval_res.apply(
    lambda r: normalize(r["pred_date"]) == normalize(r["gt_date"]), axis=1
)
df_eval_res["locality_exact"] = df_eval_res.apply(
    lambda r: normalize(r["pred_locality"]) == normalize(r["gt_locality"]), axis=1
)
df_eval_res["both_exact"]     = df_eval_res["date_exact"] & df_eval_res["locality_exact"]

n = len(df_eval_res)
date_acc     = df_eval_res["date_exact"].sum() / n
locality_acc = df_eval_res["locality_exact"].sum() / n
both_acc     = df_eval_res["both_exact"].sum() / n

print("=" * 45)
print(f" EVALUATION RESULTS  (n={n})")
print("=" * 45)
print(f" Date exact match        : {date_acc*100:.1f}%")
print(f" Locality exact match    : {locality_acc*100:.1f}%")
print(f" Both exact match        : {both_acc*100:.1f}%")
print(f" Hallucinations caught   : {df_eval_res['_hallucination_caught'].sum()}")
print(f" JSON parse errors       : {df_eval_res['_parse_error'].sum()}")
print("=" * 45)

In [ ]:
# ── AUROC & AURC (confidence calibration) ─────────────────────────────────────

def compute_aurc(correct: np.ndarray, confidence: np.ndarray) -> float:
    """
    Area Under Risk-Coverage Curve (AURC).
    Lower AURC = better (model is more confident when correct).
    """
    sorted_idx  = np.argsort(-confidence)          # descending confidence
    sorted_corr = correct[sorted_idx]
    n           = len(sorted_corr)
    risks, coverages = [], []
    for k in range(1, n + 1):
        risk     = 1 - sorted_corr[:k].mean()      # error rate at coverage k/n
        coverage = k / n
        risks.append(risk)
        coverages.append(coverage)
    aurc = np.trapz(risks, coverages)
    return aurc


date_correct  = df_eval_res["date_exact"].values.astype(float)
date_conf     = df_eval_res["pred_date_conf"].values
loc_correct   = df_eval_res["locality_exact"].values.astype(float)
loc_conf      = df_eval_res["pred_locality_conf"].values

# AUROC (only meaningful when there are both correct and incorrect)
try:
    date_auroc = roc_auc_score(date_correct, date_conf)
except ValueError:
    date_auroc = float("nan")
    print("[WARN] Date AUROC: all same class, skipping.")

try:
    loc_auroc = roc_auc_score(loc_correct, loc_conf)
except ValueError:
    loc_auroc = float("nan")
    print("[WARN] Locality AUROC: all same class, skipping.")

date_aurc = compute_aurc(date_correct, date_conf)
loc_aurc  = compute_aurc(loc_correct, loc_conf)

print(f"\nDate     → AUROC: {date_auroc:.3f}  |  AURC: {date_aurc:.3f}")
print(f"Locality → AUROC: {loc_auroc:.3f}  |  AURC: {loc_aurc:.3f}")
print("\nAUROC: 1.0 = perfect confidence discrimination")
print("AURC : lower is better (confident when correct)")

In [ ]:
# ── Visualizations ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Herbarium VLM Evaluation — Confidence Analysis", fontsize=14, fontweight="bold")

# 1. Confidence distribution split by correct/incorrect (Date)
for lbl, mask, color in [("Correct", date_correct==1, "steelblue"), ("Wrong", date_correct==0, "tomato")]:
    axes[0,0].hist(date_conf[mask.astype(bool)], bins=10, alpha=0.7, label=lbl, color=color)
axes[0,0].set_title("Date — Confidence by Correctness")
axes[0,0].set_xlabel("Confidence"); axes[0,0].set_ylabel("Count")
axes[0,0].legend()

# 2. Confidence distribution split by correct/incorrect (Locality)
for lbl, mask, color in [("Correct", loc_correct==1, "steelblue"), ("Wrong", loc_correct==0, "tomato")]:
    axes[0,1].hist(loc_conf[mask.astype(bool)], bins=10, alpha=0.7, label=lbl, color=color)
axes[0,1].set_title("Locality — Confidence by Correctness")
axes[0,1].set_xlabel("Confidence"); axes[0,1].set_ylabel("Count")
axes[0,1].legend()

# 3. Risk-Coverage curve (Date)
def risk_coverage_points(correct, confidence):
    idx  = np.argsort(-confidence)
    corr = correct[idx]
    cov  = np.arange(1, len(corr)+1) / len(corr)
    risk = 1 - np.cumsum(corr) / np.arange(1, len(corr)+1)
    return cov, risk

cov_d, risk_d = risk_coverage_points(date_correct, date_conf)
axes[1,0].plot(cov_d, risk_d, color="steelblue", linewidth=2, label=f"AURC={date_aurc:.3f}")
axes[1,0].set_title("Date — Risk-Coverage Curve")
axes[1,0].set_xlabel("Coverage"); axes[1,0].set_ylabel("Risk (Error Rate)")
axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

cov_l, risk_l = risk_coverage_points(loc_correct, loc_conf)
axes[1,1].plot(cov_l, risk_l, color="darkorange", linewidth=2, label=f"AURC={loc_aurc:.3f}")
axes[1,1].set_title("Locality — Risk-Coverage Curve")
axes[1,1].set_xlabel("Coverage"); axes[1,1].set_ylabel("Risk (Error Rate)")
axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plot_path = OUTPUT_DIR / "eval_confidence_analysis.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n📊 Plot saved → {plot_path}")

In [ ]:
# ── Print worst failures ───────────────────────────────────────────────────────
print("\n🔴 LOCALITY FAILURES (sample of up to 10):")
failures = df_eval_res[~df_eval_res["locality_exact"]][["image_file", "gt_locality", "pred_locality", "pred_locality_conf"]]
print(failures.head(10).to_string(index=False))

---
## Cell 9 — Production Inference on ~3300 Unlabeled Images
With checkpointing — safe to interrupt and resume.

In [ ]:
# ── Load test CSV ─────────────────────────────────────────────────────────────
df_test = pd.read_csv(TEST_CSV)
df_test.columns = df_test.columns.str.strip()

# Filter to existing images
df_test["_exists"] = df_test["image_file"].apply(lambda f: (IMAGES_DIR / f).exists())
missing_ct = (~df_test["_exists"]).sum()
if missing_ct > 0:
    print(f"[WARN] {missing_ct} test images not found — they will be skipped.")
df_test = df_test[df_test["_exists"]].reset_index(drop=True)

print(f"Total test images to process: {len(df_test)}")

# ── Checkpoint: resume from previous run ─────────────────────────────────────
PROD_OUTPUT = OUTPUT_DIR / "production_predictions.csv"
already_done = set()
if PROD_OUTPUT.exists():
    df_done = pd.read_csv(PROD_OUTPUT)
    already_done = set(df_done["image_file"].tolist())
    print(f"Resuming: {len(already_done)} images already processed.")

df_todo = df_test[~df_test["image_file"].isin(already_done)].reset_index(drop=True)
print(f"Remaining: {len(df_todo)} images.")

In [ ]:
# ── Production inference loop ─────────────────────────────────────────────────
CHECKPOINT_EVERY = 50    # save progress every N images

prod_results = []
start_time   = time.time()

for i, row in tqdm(df_todo.iterrows(), total=len(df_todo), desc="Production inference"):
    img_path = IMAGES_DIR / row["image_file"]
    pred     = run_inference(img_path)

    prod_results.append({
        "image_file":                    row["image_file"],
        "verbatimDate":                  pred.get("verbatimDate", "MISSING"),
        "verbatimDate_confidence":       pred.get("verbatimDate_confidence", 0.0),
        "verbatimLocality":              pred.get("verbatimLocality", "MISSING"),
        "verbatimLocality_confidence":   pred.get("verbatimLocality_confidence", 0.0),
    })

    # Periodic checkpoint save
    if (len(prod_results) % CHECKPOINT_EVERY) == 0:
        df_checkpoint = pd.DataFrame(prod_results)
        if PROD_OUTPUT.exists():
            df_existing = pd.read_csv(PROD_OUTPUT)
            df_checkpoint = pd.concat([df_existing, df_checkpoint], ignore_index=True)
        df_checkpoint.to_csv(PROD_OUTPUT, index=False)

        elapsed = time.time() - start_time
        rate    = len(prod_results) / elapsed
        remain  = (len(df_todo) - len(prod_results)) / rate if rate > 0 else 0
        print(f"  [{len(prod_results)}/{len(df_todo)}] "
              f"Rate: {rate:.1f} img/s | ETA: {remain/60:.0f} min")

# Final save
df_final = pd.DataFrame(prod_results)
if PROD_OUTPUT.exists() and len(already_done) > 0:
    df_existing = pd.read_csv(PROD_OUTPUT)
    df_final    = pd.concat([df_existing, df_final], ignore_index=True)
df_final.to_csv(PROD_OUTPUT, index=False)

total_time = time.time() - start_time
print(f"\n✅ Done! {len(df_final)} predictions saved → {PROD_OUTPUT}")
print(f"Total time: {total_time/60:.1f} min | Avg: {total_time/max(len(prod_results),1):.1f}s/image")

---
## Cell 10 — Production Results Summary & QC

In [ ]:
df_prod = pd.read_csv(OUTPUT_DIR / "production_predictions.csv")
n_total = len(df_prod)

date_missing     = (df_prod["verbatimDate"]     == "MISSING").sum()
locality_missing = (df_prod["verbatimLocality"] == "MISSING").sum()
low_date_conf    = (df_prod["verbatimDate_confidence"]     < 0.5).sum()
low_loc_conf     = (df_prod["verbatimLocality_confidence"] < 0.5).sum()

print("=" * 45)
print(f" PRODUCTION QC SUMMARY  (n={n_total})")
print("=" * 45)
print(f" Date MISSING          : {date_missing} ({date_missing/n_total*100:.1f}%)")
print(f" Locality MISSING      : {locality_missing} ({locality_missing/n_total*100:.1f}%)")
print(f" Low date confidence   : {low_date_conf} (<0.5)")
print(f" Low locality conf     : {low_loc_conf} (<0.5)")
print("=" * 45)

print("\nSample predictions:")
df_prod.head(10)[["image_file", "verbatimDate", "verbatimDate_confidence",
                   "verbatimLocality", "verbatimLocality_confidence"]]

In [ ]:
# ── Confidence distribution plot ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Production Confidence Distributions", fontsize=13, fontweight="bold")

axes[0].hist(df_prod["verbatimDate_confidence"], bins=20, color="steelblue", edgecolor="white")
axes[0].set_title("Date Confidence")
axes[0].set_xlabel("Confidence"); axes[0].set_ylabel("Count")
axes[0].axvline(0.5, color="red", linestyle="--", label="0.5 threshold")
axes[0].legend()

axes[1].hist(df_prod["verbatimLocality_confidence"], bins=20, color="darkorange", edgecolor="white")
axes[1].set_title("Locality Confidence")
axes[1].set_xlabel("Confidence"); axes[1].set_ylabel("Count")
axes[1].axvline(0.5, color="red", linestyle="--", label="0.5 threshold")
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "production_confidence_dist.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Cell 11 — VRAM & Throughput Estimation Guide

Reference table for choosing inference pixel budget on RTX 4060 8GB (4-bit Qwen-VL-7B/8B):

| max_pixels setting | Approx px | Expected VRAM | Notes |
|---|---|---|---|
| 256 × 28 × 28 | ~200K | ~5.5 GB | Very safe, fast (~4s/img) |
| **512 × 28 × 28** | **~400K** | **~6.5 GB** | **Recommended for inference** |
| 640 × 28 × 28 | ~500K | ~7.2 GB | Monitor closely |
| 768 × 28 × 28 | ~600K | ~7.8 GB | Risk of OOM on some images |
| 1024 × 28 × 28 | ~800K | OOM | Not feasible on 8 GB |

**Throughput estimate** (512×28×28, batch=1, greedy decode):
- ~4–8 seconds / image
- 3300 images → ~4–7 hours

**Tips to speed up:**
- Use `torch.compile(model)` (PyTorch 2.x) for ~20% speedup
- Use `flash_attention_2` if your CUDA version supports it
- Reduce `max_new_tokens` to 128 if JSON output is consistently short

In [ ]:
# ── Live VRAM snapshot ────────────────────────────────────────────────────────
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {allocated:.2f} GB allocated | {reserved:.2f} GB reserved | {total:.1f} GB total")
    print(f"Free headroom: ~{total - reserved:.2f} GB")

---
## Cell 12 — Optional: Flash Attention 2 + torch.compile (Speed Boost)

In [ ]:
# Flash Attention 2 is now configured in Cell 4 (model load).
# Set USE_FLASH_ATTN = True there and re-run to enable it.
print("See Cell 4 (model load) to enable Flash Attention 2.")


---
## Cell 13 — Inspect Failures Interactively

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

# Show the first N locality failure cases with the image
df_failures = df_eval_res[~df_eval_res["locality_exact"]].head(6)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (_, row) in enumerate(df_failures.iterrows()):
    img_path = IMAGES_DIR / row["image_file"]
    img = load_image(img_path)
    if img:
        axes[i].imshow(img)
    axes[i].set_title(
        f"{row['image_file']}\n"
        f"GT: {row['gt_locality']}\n"
        f"Pred: {row['pred_locality']} (conf={row['pred_locality_conf']:.2f})",
        fontsize=8
    )
    axes[i].axis("off")

# Hide unused subplots
for j in range(len(df_failures), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "locality_failures.png", dpi=120, bbox_inches="tight")
plt.show()

---
## Summary

| Step | Output |
|------|--------|
| Eval (50 images) | `outputs/eval_predictions.csv` |
| Confidence plots | `outputs/eval_confidence_analysis.png` |
| Failure cases | `outputs/locality_failures.png` |
| Production (3300) | `outputs/production_predictions.csv` |
| Production conf dist | `outputs/production_confidence_dist.png` |

### Key decisions made
- **4-bit NF4 + double quant** → fits Qwen-VL-7B in ~5.5–6.5 GB VRAM
- **512×28×28 pixel budget** → best quality/VRAM tradeoff for inference
- **Greedy decode** (`do_sample=False`) → deterministic, best for extraction tasks
- **Hallucination post-filter** → regex catches museum/university/collector text
- **Checkpointing every 50 images** → safe to interrupt long production runs